# Preprocessing

In [2]:
from datetime import datetime
import ast
import pandas as pd
from collections import Counter, defaultdict
from pathlib import Path

In [3]:
data_root = Path("../data")
hackathon_df = pd.read_csv(data_root / "processed_hackathons.csv")
projects_df = pd.read_csv(data_root / "projects.csv")

hackathon_df["submission_start"] = pd.to_datetime(
    hackathon_df["submission_start"], errors="coerce"
)

full_df = pd.merge(left=hackathon_df, right=projects_df, left_on="id", right_on="hackathon_id", how="inner")
full_df.head()

,id,title_x,url,submission_period_dates,themes,registrations_count,submission_start,submission_end,geo_location,coordinate,locality,title_y,description,built-with,is-winner,full-description,hackathon_id
0,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264,2025-11-14,2025-12-31,Online,NaN,NaN,PathFinder,a vision board that grows as you do,"css3,html5,javascript,react",False,inspiration pathfinder was inspired by a simpl...,27498
1,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264,2025-11-14,2025-12-31,Online,NaN,NaN,AdmitPlus.AI,an ai powered global admissions os that matche...,"gemini,mongodb,node.js,python,redis",False,inspiration applying to universities worldwide...,27498
2,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264,2025-11-14,2025-12-31,Online,NaN,NaN,NeuroNotes,an ai driven meeting intelligence layer for re...,"express.js,firebase,google-notebook,groq,mongo...",True,neuro notes the cognitive neural layer for syn...,27498
3,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264,2025-11-14,2025-12-31,Online,NaN,NaN,Book-puzzle,book puzzle is an interactive tool blending pu...,"css3,github,html5,javascript",False,inspiration the project was born from a collab...,27498
4,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264,2025-11-14,2025-12-31,Online,NaN,NaN,Learn. Practice. Build.,learn javascript react web machine learning is...,"html5,javascript,node.js",False,NaN,27498


## Wordcloud

In [15]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from wordcloud import WordCloud
from pathlib import Path

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

STOP_WORDS = set(stopwords.words("english"))

custom_stopwords = {
    "the", "and", "for", "with", "that", "this", "from", "into", "our", "your",
    "was", "were", "are", "have", "has", "had", "can", "will", "not", "but",
    "project", "projects", "app", "using", "used", "build", "built", "user",
    "users", "team" "make", "made", "in"
    "one", "two", "many", "would", "could", "should", "also", "like", "get", "got",
    "use", "you", "their", "one", "more", "you", "inspiration", "challenges", "challenge", "accomplishments",
    "proud", "learned", "next", "text", "without", "wanted", "nan", "ran", "di"
}


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/yumengliu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/yumengliu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/yumengliu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [16]:
WORD_CLOUD_OUTPUT_ROOT = data_root / Path("word_clouds")
if not WORD_CLOUD_OUTPUT_ROOT.exists():
    WORD_CLOUD_OUTPUT_ROOT.mkdir()

word_cloud_output = []
for year, year_df in full_df.groupby(full_df['submission_start'].dt.year):
    word_freq = Counter()
    for description in year_df["full-description"]:
        if not isinstance(description, str):
            description = str(description)
        tokens = word_tokenize(description)
        word_freq.update(tokens)

    filtered_counts = {
        word: count
        for word, count in word_freq.items()
        if word not in STOP_WORDS and word not in custom_stopwords and len(word) >= 2
    }

    top_words = Counter(filtered_counts).most_common(100)
    wc = WordCloud(width=1980, height=1080, background_color="white", max_font_size=400, min_font_size=10)
    wc.generate_from_frequencies(dict(top_words))
    img = wc.to_image()
    img.save(WORD_CLOUD_OUTPUT_ROOT / f"word_cloud_{year}.png")

    for word, count in top_words:
        word_cloud_output.append({
            "year": year,
            "word": word,
            "count": count
        })

pd.DataFrame(word_cloud_output).to_csv(data_root / "word_cloud_output.csv", index=False)

## Tool Trend

In [17]:
result = []

for year, group in full_df.groupby(full_df['submission_start'].dt.year):
    tools_counter = defaultdict(int)

    for tool_str in group["built-with"]:
        if not isinstance(tool_str, str):
            continue
        parsed_tools = tool_str.split(",")

        for tool in parsed_tools:
            tools_counter[tool.strip()] += 1

    for tool, count in tools_counter.items():
        result.append({
            "period": datetime(year, 1, 1),
            "tool": tool,
            "count": count
        })

pd.DataFrame(result).to_csv(data_root / "tool_trend.csv", index=False)

## Hackathon Themes

In [4]:
hackathon_themes = set()
for themes in hackathon_df["themes"]:
    themes = ast.literal_eval(themes)
    for theme in themes:
        hackathon_themes.add(theme["name"])

hackathon_themes

{'AR/VR',
 'Beginner Friendly',
 'Blockchain',
 'COVID-19',
 'Communication',
 'Cybersecurity',
 'Databases',
 'Design',
 'DevOps',
 'E-commerce/Retail',
 'Education',
 'Enterprise',
 'Fintech',
 'Gaming',
 'Health',
 'IoT',
 'Lifehacks',
 'Low/No Code',
 'Machine Learning/AI',
 'Mobile',
 'Music/Art',
 'Open Ended',
 'Productivity',
 'Quantum',
 'Robotic Process Automation',
 'Serverless',
 'Social Good',
 'Voice skills',
 'Web'}

In [5]:
result = []
for year, group in hackathon_df.groupby(hackathon_df['submission_start'].dt.year):
    theme_counter = {
        theme: 0
        for theme in hackathon_themes
    }
    for hackathon_theme in group["themes"]:
        hackathon_theme = ast.literal_eval(hackathon_theme)
        for theme in hackathon_theme:
            theme_counter[theme["name"]] += 1

    for theme, count in theme_counter.items():
        result.append({
            "period": datetime(year, 1, 1),
            "theme": theme,
            "count": count
        })

pd.DataFrame(result).to_csv(data_root / "theme_trend.csv", index=False)

## Locations

In [6]:
location_output = []
for year, group in hackathon_df.groupby(hackathon_df['submission_start'].dt.year):
    location_counter = defaultdict(int)

    for location in group["locality"]:
        if not isinstance(location, str):
            continue
        location_counter[location] += 1

    for location, count in location_counter.items():
        location_output.append({
            "period": datetime(year, 1, 1),
            "location": location,
            "count": count
        })

pd.DataFrame(location_output).to_csv(data_root / "location_trend.csv", index=False)